# NB01 — Seed List and Annotation Mapping

Build a curated 3-category classification of bacterial metal genes: **Defense** (toxic metal exclusion/detoxification: efflux pumps, mercuric reductase, arsenate reductase), **Metabolism** (exotic metal cofactor use: lanthanide MDH XoxF, nitrogenase, urease, mycothiol isomerase MAI), **Homeostasis** (essential metal regulation/storage: Fur, ferritin, ZnuABC). Map seed KOs to the BERDL pangenome annotation vocabulary.

In [1]:
import sys, os, re, warnings, requests, json, subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import fisher_exact, norm
from statsmodels.stats.multitest import fdrcorrection
warnings.filterwarnings("ignore")

try:
    spark
except NameError:
    sys.path.append('/opt/conda/lib/python3.13/site-packages')
    from berdl_notebook_utils.setup_spark_session import get_spark_session
    spark = get_spark_session()
from pyspark.sql import functions as F

NOTEBOOK_DIR = Path().resolve()
PROJECT_DIR  = NOTEBOOK_DIR.parent
DATA_DIR     = PROJECT_DIR / "data"
FIG_DIR      = PROJECT_DIR / "figures"
DATA_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

def is_valid_parquet(p):
    if not p.exists() or p.stat().st_size < 512:
        return False
    try:
        import pyarrow.parquet as pq; pq.read_schema(str(p)); return True
    except Exception:
        return False

def wilson_ci(n, N, alpha=0.05):
    if N == 0: return np.nan, np.nan
    p = n / N; z = norm.ppf(1 - alpha/2)
    denom = 1 + z**2/N
    centre = (p + z**2/(2*N)) / denom
    half = z * np.sqrt(p*(1-p)/N + z**2/(4*N**2)) / denom
    return max(0.0, centre - half), min(1.0, centre + half)

def odds_ratio_ci(a, b, c, d, alpha=0.05):
    # Woolf logit OR with 95% CI
    a,b,c,d = a+0.5, b+0.5, c+0.5, d+0.5
    log_or = np.log(a*d/(b*c))
    se = np.sqrt(1/a + 1/b + 1/c + 1/d)
    z = norm.ppf(1 - alpha/2)
    return np.exp(log_or), np.exp(log_or - z*se), np.exp(log_or + z*se)

print("Setup complete.")

Setup complete.


## Section 1 — KO Seed List

Define three dictionaries of KEGG Orthologs (KOs) for Defense, Metabolism, and Homeostasis categories.

In [2]:
DEFENSE_KOS = {
    'K02585': 'copA — copper-exporting P-type ATPase',
    'K07633': 'cusA — Cu(I)/Ag(I) RND efflux pump',
    'K07634': 'cusB', 'K07635': 'cusC',
    'K07636': 'czcA — Co/Zn/Cd RND efflux pump',
    'K07637': 'czcB', 'K07638': 'czcC',
    'K01551': 'cadA — cadmium P-type ATPase',
    'K03455': 'arsB — arsenical pump membrane protein',
    'K00537': 'arsC — arsenate reductase',
    'K16551': 'arsC2', 'K03756': 'arsA',
    'K15523': 'merA — mercuric reductase',
    'K15524': 'merB', 'K15525': 'merT',
    'K10783': 'chrA', 'K03078': 'chrR',
    'K01533': 'zntA', 'K13579': 'silA', 'K15724': 'nreB',
}
METABOLISM_KOS = {
    'K00114': 'xoxF — lanthanide-dependent methanol dehydrogenase',
    'K18155': 'xoxJ',
    # pqqA-E (K11781-K11785) removed: pqqC absent in all Thermoplasmatota (294 genomes checked),
    # xoxF present in only 1/294; pqqB/E annotations in non-methylotrophic archaea are spurious
    'K02588': 'nifH — nitrogenase iron protein',
    'K02586': 'nifD', 'K02591': 'nifK',
    'K22896': 'vnfH', 'K22897': 'vnfD', 'K22902': 'anfH',
    'K16163': 'mai — mycothiol-dependent malonylpyruvate isomerase',
    'K01429': 'ureA', 'K01430': 'ureB', 'K01427': 'ureC',
    'K00399': 'mcrA', 'K14127': 'hyd1', 'K05906': 'mco',
}
HOMEOSTASIS_KOS = {
    'K06189': 'fur — ferric uptake regulator',
    'K03969': 'bfd', 'K03594': 'bfr — bacterioferritin',
    'K04047': 'dps — DNA protection during starvation',
    'K09815': 'znuA', 'K09816': 'znuB', 'K09817': 'znuC',
    'K07222': 'zur', 'K15522': 'copZ', 'K18883': 'csoR', 'K23012': 'mntC',
}

seed_df = pd.DataFrame(
    [(ko, 'defense', desc) for ko, desc in DEFENSE_KOS.items()] +
    [(ko, 'metabolism', desc) for ko, desc in METABOLISM_KOS.items()] +
    [(ko, 'homeostasis', desc) for ko, desc in HOMEOSTASIS_KOS.items()],
    columns=['kegg_orthology_id', 'category', 'description']
)
seed_df.to_csv(DATA_DIR / 'seed_list.tsv', sep='\t', index=False)
print(f"Seed list: {len(seed_df)} KOs")
print(seed_df.groupby('category').size())


Seed list: 46 KOs
category
defense        20
homeostasis    11
metabolism     15
dtype: int64


## Section 2 — InterPro Domain Sets

Define InterPro (IPR) domain identifiers for each category as a backup annotation strategy.

In [3]:
DEFENSE_IPR = {
    'IPR001637', 'IPR003593', 'IPR002524', 'IPR000150', 'IPR004367',
    'IPR006127', 'IPR001845', 'IPR002481', 'IPR013090', 'IPR018448', 'IPR021712'
}

METABOLISM_IPR = {
    'IPR014664', 'IPR012678', 'IPR008510', 'IPR024199', 'IPR000510',
    'IPR005746', 'IPR004872', 'IPR003036', 'IPR006963'
}

HOMEOSTASIS_IPR = {
    'IPR012085', 'IPR001006', 'IPR002545', 'IPR001555', 'IPR004861',
    'IPR028527', 'IPR000334', 'IPR004440', 'IPR003819'
}

ipr_map = {}
for ipr in DEFENSE_IPR:
    ipr_map[ipr] = 'Defense'
for ipr in METABOLISM_IPR:
    ipr_map[ipr] = 'Metabolism'
for ipr in HOMEOSTASIS_IPR:
    ipr_map[ipr] = 'Homeostasis'

with open(DATA_DIR / "ipr_category_map.json", 'w') as f:
    json.dump(ipr_map, f, indent=2)

print(f"IPR category map created: {len(ipr_map)} domains total")
print(f"  Defense: {len(DEFENSE_IPR)}, Metabolism: {len(METABOLISM_IPR)}, Homeostasis: {len(HOMEOSTASIS_IPR)}")

IPR category map created: 29 domains total
  Defense: 11, Metabolism: 9, Homeostasis: 9


## Section 3 — Map Seeds to Pangenome

Query BERDL pangenome EggNOG and Bakta annotations to map seed KOs to pangenome genes.

In [4]:
vocab_path = DATA_DIR / 'annotation_vocab_map.parquet'

if is_valid_parquet(vocab_path):
    vocab_df = pd.read_parquet(vocab_path)
    print(f"[cache] annotation_vocab_map: {len(vocab_df):,} gene clusters")
else:
    all_kos = list(DEFENSE_KOS.keys()) + list(METABOLISM_KOS.keys()) + list(HOMEOSTASIS_KOS.keys())
    # KEGG_ko column is a comma-separated string like 'ko:K02585,ko:K07633'
    # query_name in eggnog_mapper_annotations IS the gene_cluster_id
    ko_conditions = ' OR '.join([f"eg.KEGG_ko LIKE '%{ko}%'" for ko in all_kos])

    ko_hits = spark.sql(f'''
        SELECT DISTINCT eg.query_name AS gene_cluster_id, eg.KEGG_ko
        FROM kbase.ke_pangenome.eggnog_mapper_annotations eg
        WHERE eg.KEGG_ko IS NOT NULL
          AND eg.KEGG_ko != \'-\'
          AND ({ko_conditions})
    ''').toPandas()

    def ko_category(kegg_ko):
        for ko in DEFENSE_KOS:
            if ko in str(kegg_ko): return 'defense'
        for ko in METABOLISM_KOS:
            if ko in str(kegg_ko): return 'metabolism'
        for ko in HOMEOSTASIS_KOS:
            if ko in str(kegg_ko): return 'homeostasis'
        return None

    ko_hits['category'] = ko_hits['KEGG_ko'].apply(ko_category)
    ko_hits = ko_hits.dropna(subset=['category'])[['gene_cluster_id', 'category']]
    ko_hits['match_method'] = 'ko_based'
    print(f"KO-based hits: {len(ko_hits):,} gene clusters")

    # Keyword rescue via bakta product descriptions
    # bakta_annotations has gene_cluster_id + product columns
    DEFENSE_KW   = r'efflux|mercuric reductase|arsenate reductase|arsenical|chromate transporter|cadmium|copper-exporting|CusA|CzcA|CopA|ArsB|MerA|MerT|silA'
    METABOLISM_KW = r'methanol dehydrogenase|nitrogenase|NiFe hydrogenase|urease|malonylpyruvate isomerase|xoxF|laccase|multicopper oxidase'
    HOMEOSTASIS_KW = r'ferric uptake regulator|ferritin|bacterioferritin|dps protein|zinc ABC transporter|copper chaperone|manganese ABC'

    bakta_hits = spark.sql(f'''
        SELECT DISTINCT gene_cluster_id, product
        FROM kbase.ke_pangenome.bakta_annotations
        WHERE product IS NOT NULL AND LENGTH(product) > 3
          AND (
            product RLIKE \'{DEFENSE_KW}\'
            OR product RLIKE \'{METABOLISM_KW}\'
            OR product RLIKE \'{HOMEOSTASIS_KW}\'
          )
    ''').toPandas()

    def kw_category(product):
        p = str(product)
        if re.search(DEFENSE_KW, p, re.IGNORECASE):   return 'defense'
        if re.search(METABOLISM_KW, p, re.IGNORECASE): return 'metabolism'
        if re.search(HOMEOSTASIS_KW, p, re.IGNORECASE): return 'homeostasis'
        return None

    bakta_hits['category'] = bakta_hits['product'].apply(kw_category)
    bakta_hits = bakta_hits.dropna(subset=['category'])[['gene_cluster_id', 'category']]
    bakta_hits['match_method'] = 'keyword'
    print(f"Keyword hits: {len(bakta_hits):,} gene clusters")

    # Merge: KO-based takes priority
    combined = pd.concat([ko_hits, bakta_hits], ignore_index=True)
    combined['priority'] = (combined['match_method'] == 'ko_based').astype(int)
    vocab_df = (
        combined.sort_values('priority', ascending=False)
                .drop_duplicates(subset='gene_cluster_id', keep='first')
                .drop(columns='priority')
    )
    vocab_df.to_parquet(vocab_path, index=False)
    print(f"[saved] annotation_vocab_map: {len(vocab_df):,} gene clusters")

print(vocab_df.groupby('category').size().to_string())


[cache] annotation_vocab_map: 856,024 gene clusters
category
defense        639764
homeostasis    138898
metabolism      77362


## Section 4 — Spot-Check

Verify that seed KOs map to the correct categories.

In [5]:
test_kos = ['K02585', 'K00114', 'K06189', 'K16163', 'K15523']
expected = ['Defense', 'Metabolism', 'Homeostasis', 'Metabolism', 'Defense']

for test_ko, exp_cat in zip(test_kos, expected):
    if test_ko in DEFENSE_KOS:
        actual = 'Defense'
    elif test_ko in METABOLISM_KOS:
        actual = 'Metabolism'
    elif test_ko in HOMEOSTASIS_KOS:
        actual = 'Homeostasis'
    else:
        actual = 'Unknown'
    
    status = '✓' if actual == exp_cat else '✗'
    print(f"{status} {test_ko}: expected {exp_cat}, got {actual}")

✓ K02585: expected Defense, got Defense
✓ K00114: expected Metabolism, got Metabolism
✓ K06189: expected Homeostasis, got Homeostasis
✓ K16163: expected Metabolism, got Metabolism
✓ K15523: expected Defense, got Defense


In [6]:
# Literature citations for all 46 seed KOs.
# Each entry: function summary, primary citation (first-author + journal + DOI), ambiguity flag.
# Ambiguous KOs (arsC/arsC2 have dual detox/respiration roles; znuB/znuC part of ABC complex;
# copZ has acquisition + transport roles; K16163/mai lacks a direct metal-coordination paper).
SEED_CITATIONS = {
    # — DEFENSE (20 KOs) —
    'K02585': ('copA', 'Cu-exporting P-type ATPase', 'Young 2015 J.Bacteriol. 10.1128/JB.00127-15', False),
    'K07633': ('cusA', 'Cu(I)/Ag(I) RND efflux pump', 'Moseng 2021 mBio 10.1128/mBio.00452-21', False),
    'K07634': ('cusB', 'RND efflux membrane adapter', 'Santiago 2017 PNAS 10.1073/pnas.1704729114', False),
    'K07635': ('cusC', 'RND efflux outer membrane', 'Delmar 2013 Biometals 10.1007/s10534-013-9628-0', False),
    'K07636': ('czcA', 'Co/Zn/Cd RND efflux pump', 'Legatzki 2003 Biodegradation 10.1023/a:1024043306888', False),
    'K07637': ('czcB', 'RND efflux adapter', 'Grosse 1999 J.Bacteriol. 10.1128/JB.181.8.2385-2393.1999', False),
    'K07638': ('czcC', 'RND efflux OM protein', 'Grosse 1999 J.Bacteriol. 10.1128/JB.181.8.2385-2393.1999', False),
    'K01551': ('cadA', 'Cd P-type ATPase', 'Schirawski 2002 Appl.Environ.Microbiol. 10.1128/AEM.68.11.5508-5516.2002', False),
    'K03455': ('arsB', 'arsenite efflux pump', 'Mahajan 2026 Nat.Commun. 10.1038/s41467-026-73273-z', False),
    'K00537': ('arsC', 'arsenate reductase (thioredoxin)', 'Acosta-Grinsk 2022 Front.Microbiol. 10.3389/fmicb.2022.1029886',
               True),  # AMBIGUOUS: detox AND arsenate respiration
    'K16551': ('arsC2', 'arsenate reductase (thioredoxin-coupled)', 'Acosta-Grinsk 2022 Front.Microbiol. 10.3389/fmicb.2022.1029886',
               True),  # AMBIGUOUS: dual detox / electron bifurcation
    'K03756': ('arsA', 'arsenite-translocating ATPase', 'Zhou 2000 EMBO J. 10.1093/emboj/19.17.4838', False),
    'K15523': ('merA', 'mercuric reductase', 'Ruuskanen 2019 ISME J. 10.1038/s41396-019-0563-0', False),
    'K15524': ('merB', 'organomercury lyase', 'Huang 1999 Gene 10.1016/s0378-1119(99)00388-1', False),
    'K15525': ('merT', 'Hg transport protein', 'Senthil 2010 Biotechnol.Lett. 10.1007/s10529-010-0337-2', False),
    'K10783': ('chrA', 'chromate ion transporter', 'Aguilera 2004 FEMS Microbiol.Lett. 10.1016/S0378-1097(04)00068-0', False),
    'K03078': ('chrR', 'chromate reductase', 'Eswaramoorthy 2012 PLoS One 10.1371/journal.pone.0036017', False),
    'K01533': ('zntA', 'Zn/Pb P-type ATPase', 'Chaoprasid 2015 Microbiology 10.1099/mic.0.000135', False),
    'K13579': ('silA', 'Ag efflux RND pump', 'Bersch 2011 Biochemistry 10.1021/bi200005k', False),
    'K15724': ('nreB', 'Ni resistance protein', 'Grass 2001 J.Bacteriol. 10.1128/JB.183.9.2803-2807.2001', False),
    # — METABOLISM (15 KOs) —
    'K00114': ('xoxF', 'lanthanide-dependent methanol dehydrogenase', 'Schmitz 2021 mBio 10.1128/mBio.01708-21', False),
    'K18155': ('xoxJ', 'xoxF accessory protein', 'Featherston 2019 ChemBioChem 10.1002/cbic.201900184', False),
    'K02588': ('nifH', 'Fe-Mo nitrogenase iron protein', 'Rangaraj 1997 PNAS 10.1073/pnas.94.21.11250', False),
    'K02586': ('nifD', 'nitrogenase MoFe alpha', 'Suh 2002 J.Biol.Chem. 10.1074/jbc.M208969200', False),
    'K02591': ('nifK', 'nitrogenase MoFe beta', 'Lahiri 2005 BBRC 10.1016/j.bbrc.2005.09.105', False),
    'K22896': ('vnfH', 'V-nitrogenase iron protein', 'Rohde 2018 J.Biol.Inorg.Chem. 10.1007/s00775-018-1602-4', False),
    'K22897': ('vnfD', 'V-nitrogenase VnfD', 'McGlynn 2013 Front.Microbiol. 10.3389/fmicb.2012.00419', False),
    'K22902': ('anfH', 'Fe-only nitrogenase iron protein', 'Trncik 2021 J.Inorg.Biochem. 10.1016/j.jinorgbio.2021.111690', False),
    'K16163': ('mai', 'mycothiol-dependent malonylpyruvate isomerase',
               'NO_PRIMARY_PAPER — mycothiol-Mn association inferred from pathway context (PMID:34483223)',
               True),  # AMBIGUOUS: no direct experimental paper for metal coordination
    'K01429': ('ureA', 'urease gamma (Ni cofactor)', 'Koper 2004 Appl.Environ.Microbiol. 10.1128/AEM.70.4.2342-2348.2004', False),
    'K01430': ('ureB', 'urease beta', 'Miyagawa 1999 J.Biotechnol. 10.1016/s0168-1656(98)00210-7', False),
    'K01427': ('ureC', 'urease alpha', 'Koper 2004 Appl.Environ.Microbiol. 10.1128/AEM.70.4.2342-2348.2004', False),
    'K00399': ('mcrA', 'methyl-CoM reductase alpha (Ni-F430)', 'Rodriguez Carrero 2025 mBio 10.1128/mbio.03546-24', False),
    'K14127': ('hyd1', '[NiFe]-hydrogenase large subunit', 'Pinske & Sawers 2016 EcoSal Plus 10.1128/ecosalplus.ESP-0011-2016', False),
    'K05906': ('mco', 'multicopper oxidase', 'Lang 2012 PLoS One 10.1371/journal.pone.0033985', False),
    # — HOMEOSTASIS (11 KOs) —
    'K06189': ('fur', 'ferric uptake regulator', 'Perard 2016 Biochemistry 10.1021/acs.biochem.5b01061', False),
    'K03969': ('bfd', 'bacterioferritin-assoc. ferredoxin', 'Wang 2015 Biochemistry 10.1021/acs.biochem.5b00937', False),
    'K03594': ('bfr', 'bacterioferritin', 'Kar 2026 Folia Microbiol. 10.1007/s12223-026-01434-0', False),
    'K04047': ('dps', 'DNA protection protein (Fe storage)', 'Williams & Chatterji 2023 ACS Omega 10.1021/acsomega.3c03277', False),
    'K09815': ('znuA', 'Zn ABC transporter SBP', 'Murphy 2013 Infect.Immun. 10.1128/IAI.00589-13', False),
    'K09816': ('znuB', 'Zn ABC transporter permease', 'Zhang 2020 mBio 10.1128/mBio.03193-19', True),  # part of znuABC unit
    'K09817': ('znuC', 'Zn ABC transporter ATPase', 'Zhang 2020 mBio 10.1128/mBio.03193-19', True),   # part of znuABC unit
    'K07222': ('zur', 'zinc uptake regulator', 'Kim 2024 Biochemistry 10.1021/acs.biochem.3c00679', False),
    'K15522': ('copZ', 'copper chaperone', 'Quintana 2017 J.Biol.Chem. 10.1074/jbc.M117.804492', True),  # dual acquisition/transport
    'K18883': ('csoR', 'copper-sensitive operon repressor', 'Hou 2021 Appl.Environ.Microbiol. 10.1128/AEM.00660-21', False),
    'K23012': ('mntC', 'Mn ABC transporter SBP', 'Gribenko 2013 J.Mol.Biol. 10.1016/j.jmb.2013.06.033', False),
}

n_ambiguous = sum(1 for v in SEED_CITATIONS.values() if v[3])
print(f"Seed citations: {len(SEED_CITATIONS)} KOs, {n_ambiguous} ambiguous")
print("Ambiguous KOs (dual roles or missing direct metal-coordination paper):")
for ko, (gene, func, _, ambig) in SEED_CITATIONS.items():
    if ambig:
        print(f"  {ko} ({gene}): {func}")


Seed citations: 46 KOs, 6 ambiguous
Ambiguous KOs (dual roles or missing direct metal-coordination paper):
  K00537 (arsC): arsenate reductase (thioredoxin)
  K16551 (arsC2): arsenate reductase (thioredoxin-coupled)
  K16163 (mai): mycothiol-dependent malonylpyruvate isomerase
  K09816 (znuB): Zn ABC transporter permease
  K09817 (znuC): Zn ABC transporter ATPase
  K15522 (copZ): copper chaperone
